# Units SAE data-scale test: 1,000 versus 10,000 prompts

This notebook tests one specific explanation for the SAE limitations: too few activation examples. It retrains the units TopK-128 SAEs on a deterministic, balanced corpus of 10,000 unique prompts and repeats the existing force-selective confirmation protocol. The enlarged corpus retains all 1,000 original units prompts and adds 9,000 controlled prompts. Model, layers, latent width, TopK value, optimiser settings, seed, graph objective, discovery/confirmation split, panel size, and intervention implementation remain fixed. Only activation-corpus scale and controlled linguistic diversity change.

This is a one-seed data-scale ablation, not a new hyperparameter search. Feature identifiers cannot be compared across independently trained dictionaries; reconstruction metrics and causal effect estimates can be compared.

## 1. Mount Drive, fetch the repository, and install it

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import shlex
import subprocess
import sys
from pathlib import Path

def run_cmd(command):
    command = [str(part) for part in command]
    print('$', shlex.join(command), flush=True)
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )
    if process.stdout is None:
        raise RuntimeError('Could not capture child-process output')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

repo_url = 'https://github.com/evey-dev/test_run.git'
repo_dir = Path('/content/test_run')
checkout = repo_dir
try:
    if (checkout / '.git').is_dir():
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        if checkout.exists() and any(checkout.iterdir()):
            checkout = Path('/content/test_run_github')
        if (checkout / '.git').is_dir():
            run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
        elif checkout.exists() and any(checkout.iterdir()):
            raise RuntimeError(f'{checkout} exists but is not a Git repository')
        else:
            run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
except Exception as exc:
    print('GitHub checkout failed; trying the Drive project.zip backup.')
    print(repr(exc))
    zip_path = Path('/content/drive/MyDrive/mphil-project/project.zip')
    if not zip_path.exists():
        raise
    run_cmd(['unzip', '-q', '-o', zip_path, '-d', '/content'])
    candidates = [Path('/content/test_run'), Path('/content/mphil_project/test_run')]
    checkout = next((path for path in candidates if (path / 'src').is_dir()), None)
    if checkout is None:
        raise FileNotFoundError('Could not locate the extracted repository root')

os.chdir(checkout)
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51,<5', 'accelerate', 'pandas<2.4', 'pyyaml', 'tqdm'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
print('Repository root:', Path.cwd())

## 2. Fixed design and artifact paths

In [ ]:
import csv
import json
import re
import shutil
import statistics

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
LAYERS = [4, 8, 12, 16, 20, 24, 28]
DATA_ARG = 'data/units_large_10000.csv'
CONFIG_ARG = 'configs/sae_units_large_10000_topk128_config.yaml'
ACTIVATION_DIR = ROOT / 'mechanistic_data_units_large_10000'
CHECKPOINT_DIR = ROOT / 'mechanistic_data/sae_checkpoints_units_large_10000_topk128'
LOCAL_OUTPUT = ROOT / 'outputs/units_large_data_test'

DRIVE_ROOT = Path('/content/drive/MyDrive/mphil-project')
DRIVE_EXPERIMENT = DRIVE_ROOT / 'mechanistic_data/units_large_data_test'
DRIVE_ACTIVATIONS = DRIVE_EXPERIMENT / 'activations'
DRIVE_CHECKPOINTS = DRIVE_EXPERIMENT / 'checkpoints_topk128'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs/units_large_data_test'
for path in (LOCAL_OUTPUT, DRIVE_EXPERIMENT, DRIVE_ACTIVATIONS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

required = [
    Path('data/generate_large_units_dataset.py'),
    Path('data/units_data.csv'),
    Path(CONFIG_ARG),
    Path('src/capture_activations.py'),
    Path('src/units_feature_screen.py'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Push the new experiment files to GitHub before running: ' + ', '.join(missing))
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing')

config = yaml.safe_load(Path(CONFIG_ARG).read_text())
assert config['activation_type'] == 'topk' and config['top_k'] == 128
assert config['layers'] == LAYERS and config['seed'] == 787
assert config.get('save_latents') is False
print('GPU:', torch.cuda.get_device_name(0))
print('Drive experiment root:', DRIVE_EXPERIMENT)

## 3. Generate and audit the 10,000-prompt corpus

The corpus retains every original row, then adds examples drawn from five quantities crossed with the same 25 apparatuses, ten operating conditions, and eight templates. It contains exactly 2,000 prompts per quantity. Exact held-out confirmation prompts remain absent.

In [ ]:
run_cmd([
    sys.executable, 'data/generate_large_units_dataset.py',
    '--count', '10000', '--seed', '787',
    '--base-csv', 'data/units_data.csv', '--output', DATA_ARG,
])
corpus = pd.read_csv(DATA_ARG)
quantity_counts = corpus.groupby('Quantity').size().sort_index()
assert len(corpus) == 10_000
assert corpus['sentence'].nunique() == 10_000
assert set(quantity_counts.tolist()) == {2_000}
original_corpus = pd.read_csv('data/units_data.csv')
assert set(original_corpus['sentence']).issubset(set(corpus['sentence']))

from data.generate_large_units_dataset import TRAINING_SYSTEMS
from src.units_feature_screen import SYSTEMS, generated_context_cases
assert set(TRAINING_SYSTEMS).isdisjoint(SYSTEMS)
confirmation_prompts = {
    prompt
    for case in generated_context_cases(seed=2787)
    for prompt in (case['force_prompt'], case['mass_prompt'], case['energy_prompt'])
}
assert set(corpus['sentence']).isdisjoint(confirmation_prompts)
display(quantity_counts.to_frame('prompts'))
display(corpus[['Quantity', 'ContextObject', 'OperatingContext', 'sentence']].head(5))

## 4. Capture final-token MLP activations

This cell restores a complete capture from Drive when available. Otherwise it captures 10,000 activation vectors at each of the seven fixed layers and immediately persists them.

In [ ]:
activation_files = [f'activations_layer{layer}.npy' for layer in LAYERS] + [
    'train_indices.npy',
    'val_indices.npy',
    'train_val_indices_per_layer.npy',
    'activation_metadata.csv',
]

def complete(directory, names):
    return all((directory / name).exists() for name in names)

if not complete(ACTIVATION_DIR, activation_files) and complete(DRIVE_ACTIVATIONS, activation_files):
    print('Restoring activation capture from Drive...')
    shutil.copytree(DRIVE_ACTIVATIONS, ACTIVATION_DIR, dirs_exist_ok=True)

if not complete(ACTIVATION_DIR, activation_files):
    run_cmd([
        sys.executable, '-m', 'src.capture_activations',
        '--model-config', 'configs/model_config.yaml',
        '--output-dir', str(ACTIVATION_DIR),
        '--prompt-csv', DATA_ARG,
        '--prompt-behaviour', 'units_large_10000',
        '--layers', *map(str, LAYERS),
        '--seed', '787',
    ])
    shutil.copytree(ACTIVATION_DIR, DRIVE_ACTIVATIONS, dirs_exist_ok=True)
else:
    print('Activation capture already complete; skipping capture.')

assert complete(ACTIVATION_DIR, activation_files)
train_indices = np.load(ACTIVATION_DIR / 'train_indices.npy')
val_indices = np.load(ACTIVATION_DIR / 'val_indices.npy')
assert len(train_indices) == 8_000 and len(val_indices) == 2_000
split_audit = pd.DataFrame({
    'train': corpus.iloc[train_indices].groupby('Quantity').size(),
    'validation': corpus.iloc[val_indices].groupby('Quantity').size(),
})
display(split_audit)

## 5. Train the fixed TopK-128 SAEs

Each completed layer is backed up to Drive and restored on rerun. Dense latent exports are disabled because they are not consumed by attribution or intervention code; this avoids approximately 2.3 GB of redundant arrays without changing training or checkpoints.

In [ ]:
run_cmd([
    sys.executable, '-m', 'src.train',
    '--config', CONFIG_ARG,
    '--drive-dir', str(DRIVE_CHECKPOINTS),
])
checkpoint_files = [
    name
    for layer in LAYERS
    for name in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json')
]
assert complete(CHECKPOINT_DIR, checkpoint_files)
print('Complete checkpoint layers:', LAYERS)
print('Dense latent arrays written:', len(list(CHECKPOINT_DIR.glob('latents_layer*.npy'))))

## 6. Measure held-out reconstruction and sparsity

In [ ]:
diagnostic_json = LOCAL_OUTPUT / 'units_large_10000_topk128_diagnostics.json'
diagnostic_csv = LOCAL_OUTPUT / 'units_large_10000_topk128_diagnostics.csv'
drive_diagnostic_json = DRIVE_OUTPUT / diagnostic_json.name
drive_diagnostic_csv = DRIVE_OUTPUT / diagnostic_csv.name
if drive_diagnostic_json.exists() and drive_diagnostic_csv.exists():
    shutil.copy2(drive_diagnostic_json, diagnostic_json)
    shutil.copy2(drive_diagnostic_csv, diagnostic_csv)
    print('Restored diagnostics from Drive.')
else:
    run_cmd([
        sys.executable, '-m', 'src.sae_diagnostics',
        '--config', CONFIG_ARG,
        '--label', 'units_large_10000_topk128',
        '--device', 'cuda',
        '--output-json', str(diagnostic_json),
        '--output-csv', str(diagnostic_csv),
    ])
    shutil.copy2(diagnostic_json, drive_diagnostic_json)
    shutil.copy2(diagnostic_csv, drive_diagnostic_csv)

diagnostics_table = pd.read_csv(diagnostic_csv)
display(diagnostics_table[[
    'layer', 'validation_fraction_variance_explained',
    'validation_mean_relative_l2_error', 'validation_mean_l0',
    'combined_dead_feature_fraction',
]])

## 7. Rebuild the fixed force-versus-energy attribution graph

The graph prompt, target-minus-contrast objective, layer set, and pruning limits match the 1,000-prompt TopK-128 experiment.

In [ ]:
GRAPH_STEM = 'units_large_10000_topk128_force_graph'
graph_paths = {suffix: LOCAL_OUTPUT / f'{GRAPH_STEM}.{suffix}' for suffix in ('json', 'html', 'md')}
drive_graph_paths = {suffix: DRIVE_OUTPUT / path.name for suffix, path in graph_paths.items()}
if all(path.exists() for path in drive_graph_paths.values()):
    for suffix in graph_paths:
        shutil.copy2(drive_graph_paths[suffix], graph_paths[suffix])
    print('Restored attribution graph from Drive.')
else:
    run_cmd([
        sys.executable, '-m', 'src.attribution_graph',
        '--prompt', 'Fact: The official SI unit for the force of a moving engine thrust is named "',
        '--target', 'newtons',
        '--contrast-target', 'joules',
        '--layers', *map(str, LAYERS),
        '--sae-config', CONFIG_ARG,
        '--top-k-nodes', '20',
        '--top-k-edges', '30',
        '--output-json', str(graph_paths['json']),
        '--output-html', str(graph_paths['html']),
        '--output-mermaid', str(graph_paths['md']),
    ])
    for suffix in graph_paths:
        shutil.copy2(graph_paths[suffix], drive_graph_paths[suffix])

graph = json.loads(graph_paths['json'].read_text())
positive_features = [
    node for node in graph['nodes']
    if re.fullmatch(r'layer_\d+_feature_\d+', str(node.get('id', '')))
    and float(node.get('attribution', 0.0)) > 0
]
print('Objective:', graph['attribution_objective'])
print('Objective value:', graph['attribution_objective_value'])
print('Positive graph feature candidates:', len(positive_features))

## 8. Secondary graph-held-out benchmark

This preserves the existing broad graph-feature benchmark as a secondary result. The compact-panel screen in the next section remains the primary causal comparison.

In [ ]:
heldout_path = LOCAL_OUTPUT / 'units_large_10000_topk128_heldout_last.json'
drive_heldout_path = DRIVE_OUTPUT / heldout_path.name
if drive_heldout_path.exists():
    shutil.copy2(drive_heldout_path, heldout_path)
    print('Restored graph-held-out benchmark from Drive.')
else:
    run_cmd([
        sys.executable, '-m', 'src.heldout_validation',
        '--units-sae-config', CONFIG_ARG,
        '--units-graph', str(graph_paths['json'].relative_to(ROOT)),
        '--unit-cases', '20',
        '--skip-math',
        '--positions', 'last',
        '--output', str(heldout_path),
    ])
    shutil.copy2(heldout_path, drive_heldout_path)
heldout = json.loads(heldout_path.read_text())
display(pd.Series(heldout['units']['summary']).to_frame('value'))

## 9. Primary compact force-selectivity screen

Candidate features come from the positive force graph. Ranking uses eight discovery systems; the fixed Top-10 panel is then evaluated on sixteen disjoint confirmation systems against a matched mass-source control. The output is checkpointed directly to Drive and resumes after interruption.

In [ ]:
drive_screen_path = DRIVE_OUTPUT / 'units_large_10000_topk128_feature_screen.json'
run_cmd([
    sys.executable, '-m', 'src.units_feature_screen',
    '--sae-config', CONFIG_ARG,
    '--sae-corpus-csv', DATA_ARG,
    '--graph', str(graph_paths['json'].relative_to(ROOT)),
    '--positions', 'last',
    '--discovery-cases', '8',
    '--confirmation-cases', '16',
    '--seed', '2787',
    '--panel-sizes', '1', '3', '5', '10', '20',
    '--primary-panel-size', '10',
    '--random-panels', '5',
    '--output', str(drive_screen_path),
])
screen_path = LOCAL_OUTPUT / drive_screen_path.name
shutil.copy2(drive_screen_path, screen_path)
screen = json.loads(screen_path.read_text())
primary = screen['confirmation']['primary_result']
print('Status:', screen['status'])
print('Candidate features:', screen['candidate_feature_count'])
print('Primary panel:', primary['panel'])
print('Predeclared success rule met:', primary['supports_force_selectivity_under_predeclared_rule'])
display(pd.Series(primary['summary']).to_frame('value'))

## 10. Direct 1,000-versus-10,000 comparison

The table compares dictionary-level diagnostics and the same predeclared Top-10 causal estimand. Higher validation FVE and lower relative error indicate improved reconstruction. A positive force-minus-mass effect with a confidence interval wholly above zero indicates force-selective transfer. Feature IDs themselves are not comparable across retraining runs.

In [ ]:
reference_diagnostics_path = ROOT / 'outputs/topk_units_retrain/units_topk128_diagnostics.json'
reference_screen_path = ROOT / 'outputs/topk_units_retrain/units_topk128_feature_screen.json'
if not reference_diagnostics_path.exists() or not reference_screen_path.exists():
    raise FileNotFoundError('The reference TopK-128 result files are missing from the checkout')

def comparison_row(label, diagnostic_path, causal_path):
    diagnostic = json.loads(Path(diagnostic_path).read_text())
    causal = json.loads(Path(causal_path).read_text())
    layers = diagnostic['layers']
    summary = causal['confirmation']['primary_result']['summary']
    first = layers[0]
    examples = first['train']['samples'] + first['validation']['samples']
    paired_ci = summary['bootstrap_95_ci_mean_force_minus_mass_difference']
    return {
        'run': label,
        'activation_examples_per_layer': examples,
        'mean_validation_fve': statistics.fmean(row['validation']['fraction_variance_explained'] for row in layers),
        'mean_validation_relative_l2': statistics.fmean(row['validation']['mean_relative_l2_error'] for row in layers),
        'mean_validation_l0': statistics.fmean(row['validation']['mean_l0'] for row in layers),
        'mean_dead_feature_fraction': statistics.fmean(row['combined_dead_feature_fraction'] for row in layers),
        'positive_graph_candidates': causal['candidate_feature_count'],
        'primary_force_delta': summary['mean_force_source_delta'],
        'primary_mass_delta': summary['mean_mass_source_delta'],
        'primary_force_minus_mass': summary['mean_force_minus_mass_difference'],
        'paired_ci_low': paired_ci[0],
        'paired_ci_high': paired_ci[1],
        'force_delta_positive_fraction': summary['fraction_force_delta_positive'],
        'force_more_positive_than_mass_fraction': summary['fraction_force_more_positive_than_mass'],
        'force_top_prediction_transfer_fraction': summary['force_top_prediction_transfer_fraction'],
        'predeclared_success': causal['confirmation']['primary_result']['supports_force_selectivity_under_predeclared_rule'],
    }

comparison_rows = [
    comparison_row('units_topk128_1k', reference_diagnostics_path, reference_screen_path),
    comparison_row('units_topk128_10k', diagnostic_json, screen_path),
]
comparison_table = pd.DataFrame(comparison_rows).set_index('run')
display(comparison_table.T)

comparison_payload = {
    'question': 'Does a larger controlled units activation corpus improve TopK-128 SAE reconstruction or force-selective causal localization?',
    'fixed_factors': [
        'Qwen3-4B-Instruct-2507', 'layers 4,8,12,16,20,24,28',
        'latent_dim 8192', 'TopK 128', 'seed 787', '75 epochs',
        'force-minus-joules graph objective', 'Top-10 confirmation panel',
    ],
    'changed_factor': 'balanced units activation corpus: original 1,000 prompts plus 9,000 controlled prompts',
    'runs': comparison_rows,
}
comparison_json = LOCAL_OUTPUT / 'units_1k_vs_10k_comparison.json'
comparison_csv = LOCAL_OUTPUT / 'units_1k_vs_10k_comparison.csv'
comparison_json.write_text(json.dumps(comparison_payload, indent=2))
comparison_table.reset_index().to_csv(comparison_csv, index=False)
shutil.copy2(comparison_json, DRIVE_OUTPUT / comparison_json.name)
shutil.copy2(comparison_csv, DRIVE_OUTPUT / comparison_csv.name)

## 11. Save provenance and verify persistent artifacts

In [ ]:
for path in LOCAL_OUTPUT.iterdir():
    if path.is_file():
        shutil.copy2(path, DRIVE_OUTPUT / path.name)
provenance_dir = DRIVE_EXPERIMENT / 'provenance'
provenance_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(DATA_ARG, provenance_dir / Path(DATA_ARG).name)
shutil.copy2(CONFIG_ARG, provenance_dir / Path(CONFIG_ARG).name)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() if (ROOT / '.git').exists() else None
manifest = {
    'experiment': 'units_large_10000_topk128',
    'git_commit': commit,
    'gpu': torch.cuda.get_device_name(0),
    'seed': 787,
    'layers': LAYERS,
    'corpus_rows': len(corpus),
    'train_rows': len(train_indices),
    'validation_rows': len(val_indices),
    'config': CONFIG_ARG,
    'dense_latent_export': False,
    'local_outputs': sorted(path.name for path in LOCAL_OUTPUT.iterdir() if path.is_file()),
    'drive_output': str(DRIVE_OUTPUT),
    'drive_checkpoints': str(DRIVE_CHECKPOINTS),
    'drive_activations': str(DRIVE_ACTIVATIONS),
}
manifest_path = LOCAL_OUTPUT / 'experiment_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
shutil.copy2(manifest_path, DRIVE_OUTPUT / manifest_path.name)
print(json.dumps(manifest, indent=2))
print('All persistent outputs are under:', DRIVE_OUTPUT)